# NB03: train, validate, export samples, publish

Builds the word-level table (each flawed clip against its control), trains the LightGBM detector, runs the automated validation on the test split, exports dashboard samples and publishes OCDD and the detector to the Hugging Face Hub.

Runtime: CPU is enough.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/oratio")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Jonathan-Jerrick/oratio.git", str(REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO}/packages/speechgrade",
        "-e",
        f"{REPO}/dataset_factory[colab]",
    ],
    capture_output=True,
    text=True,
)
print(install.stdout[-2000:], install.stderr[-4000:])
install.check_returncode()
for src in (REPO / "packages/speechgrade/src", REPO / "dataset_factory/src"):
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
print(
    subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout
)

In [ ]:
import json

from oratio_data.stages import nb03

ROOT = Path("/content/drive/MyDrive/Oratio")

In [ ]:
training = nb03.train(ROOT)
print(json.dumps(training, indent=1))

In [ ]:
validation = nb03.evaluate(ROOT)
print(json.dumps(validation, indent=1))

In [ ]:
samples = nb03.export_samples(ROOT)
print(len(samples), [s["id"] for s in samples])

In [ ]:
from google.colab import userdata

published = nb03.publish(
    ROOT, {"training": training, "validation": validation}, userdata.get("HF_TOKEN")
)
print(published)